# Sinhala Document Understanding — Donut (Kaggle GPU notebook)

**This notebook only runs on Kaggle with a GPU accelerator and the `SinFundDonut` dataset attached — it cannot run on a local CPU-only machine.** See `Guide/rules.md` for the full compatibility checklist before editing anything below.

Pick `MODEL_VARIANT` in Cell 1:

| Variant | Encoder | Decoder | Tokenizer | Purpose |
|---|---|---|---|---|
| `donut_native` | Donut Swin | Donut's own pretrained decoder | Donut's original multilingual tokenizer | Fast sanity check that the fixed data/training pipeline runs end-to-end. Sinhala falls back to inefficient byte-level tokenization. |
| `donut_sinhala_vocab` **(recommended default)** | Donut Swin | Donut's own pretrained decoder | SinBERT (Sinhala) | Keeps Donut's pretrained encoder↔decoder cross-attention intact while giving the decoder a Sinhala-efficient vocabulary. |
| `hybrid` | Donut Swin | Custom Sinhala TrOCR decoder | SinBERT (Sinhala) | The "Lego" research contribution (Option B in `Guide/README.md`). Cross-attention is untrained from scratch — try only after the two variants above produce sane, non-zero metrics. |

After each Kaggle run, save the notebook's output (`File → Download` or the Kaggle "Output" tab) into `KaggleRanNotebook/`, and paste the full cell logs into `KaggleRanNotebook/logs.txt` so the next iteration can be diagnosed from real evidence.


In [7]:
# ============================================================
# CELL 1 — ENVIRONMENT, CONFIGURATION, GPU CHECK, HF AUTH,
#          DATASET PATH AUTO-DETECTION
# ============================================================
# Kaggle-only notebook. See Guide/rules.md for the full checklist
# (GPU accelerator, Internet ON, HF_TOKEN secret, dataset attached).

import os

# Use a single GPU even if "GPU T4 x2" is selected in Kaggle, to
# avoid DataParallel / device-mismatch complications.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import glob
import json
import random

import numpy as np
import torch

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

# "donut_native"         -> Donut Swin encoder + Donut's own decoder
#                            + Donut's original tokenizer. Fast
#                            end-to-end sanity check.
# "donut_sinhala_vocab"  -> RECOMMENDED DEFAULT. Donut Swin encoder +
#                            Donut's own decoder + SinBERT tokenizer.
# "hybrid"                -> Donut Swin encoder + custom Sinhala TrOCR
#                            decoder + SinBERT tokenizer (Option B /
#                            "Lego" approach in Guide/README.md).
MODEL_VARIANT = "donut_sinhala_vocab"

TOKENIZER_ID = "NLPC-UOM/SinBERT-large"
TROCR_HUB_ID = "danush99/Model_TrOCR-Sin-Handwritten-Text"
DONUT_BASE_ID = "naver-clova-ix/donut-base"

TASK_START_TOKEN = "<s_gt_parse>"
TASK_END_TOKEN = "</s_gt_parse>"

OUTPUT_DIR = "./donut_sinhala_output"
FINAL_MODEL_PATH = "./donut_sinhala_final"

# Donut Swin requires dimensions compatible with its patch structure.
IMAGE_SIZE = [960, 640]

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# GPU CHECK (fail fast — this notebook must run on a Kaggle GPU)
# ------------------------------------------------------------

print("=" * 70)
print("GPU / PYTORCH INFORMATION")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. This notebook only runs on Kaggle "
        "with a GPU accelerator (Notebook Settings -> Accelerator -> "
        "GPU T4 x2, or similar). It is NOT meant to run on a local "
        "CPU-only machine — see Guide/rules.md."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)

# Basic CUDA test
x = torch.randn(256, 256, device="cuda")
y = torch.randn(256, 256, device="cuda")
z = x @ y
print("Basic CUDA test: PASSED")
del x, y, z
torch.cuda.empty_cache()

# ------------------------------------------------------------
# HUGGING FACE AUTHENTICATION (Kaggle Secrets)
# ------------------------------------------------------------
# Requires: Kaggle notebook -> Add-ons -> Secrets -> "HF_TOKEN"
# Requires: Notebook Settings -> Internet -> ON (needed to download
# naver-clova-ix/donut-base, NLPC-UOM/SinBERT-large and the custom
# TrOCR model from the Hugging Face Hub).

HF_TOKEN = None

try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("\nHugging Face authentication: configured")

except Exception as e:
    print(
        "\nWarning: HF_TOKEN could not be loaded from Kaggle secrets:",
        e,
        "\nPublic-only models will be used; the custom TrOCR model "
        "(hybrid variant) requires this token if the repo is private.",
    )

if HF_TOKEN:
    try:
        from huggingface_hub import whoami

        user = whoami(token=HF_TOKEN)
        print("Hugging Face user:", user.get("name") or user.get("fullname") or "unknown")
    except Exception as e:
        print("HF authentication check failed:", e)

# ------------------------------------------------------------
# DATASET PATH AUTO-DETECTION
# ------------------------------------------------------------
# Kaggle mounts attached datasets under /kaggle/input/<dataset-slug>/...
# The exact slug depends on how the dataset was uploaded/named, so we
# search for it by content instead of hardcoding a path that can
# silently go stale (this was a real bug in the previous version of
# this notebook — see Guide/learnings.md).


def find_dataset_root(marker_dirname="SinFundDonut"):
    candidates = []
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for path in glob.glob(os.path.join(root, "**", marker_dirname), recursive=True):
            if os.path.exists(os.path.join(path, "train", "metadata.jsonl")):
                candidates.append(path)
    return candidates


_candidates = find_dataset_root()

if not _candidates:
    print("\nContents of /kaggle/input (for debugging):")
    for p in glob.glob("/kaggle/input/*"):
        print(" -", p)
    raise FileNotFoundError(
        "Could not locate the SinFundDonut dataset anywhere under "
        "/kaggle/input. Attach it via Notebook Settings -> Add "
        "Input -> Datasets -> SinFundDonut."
    )

DATASET_ROOT = _candidates[0]
TRAIN_PATH = os.path.join(DATASET_ROOT, "train")
VALIDATION_PATH = os.path.join(DATASET_ROOT, "validation")

print("\nDataset root detected at:", DATASET_ROOT)
print("Train path:", TRAIN_PATH)
print("Validation path:", VALIDATION_PATH)

assert os.path.exists(TRAIN_PATH), f"Train path not found: {TRAIN_PATH}"
assert os.path.exists(VALIDATION_PATH), f"Validation path not found: {VALIDATION_PATH}"

print("=" * 70)


GPU / PYTORCH INFORMATION
PyTorch: 2.10.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
GPU: Tesla T4
Compute capability: (7, 5)
GPU memory: 14.56 GB
Basic CUDA test: PASSED

Hugging Face authentication: configured
Hugging Face user: danush99

Dataset root detected at: /kaggle/input/datasets/danushamsc25/sinfunddonut/SinFundDonut
Train path: /kaggle/input/datasets/danushamsc25/sinfunddonut/SinFundDonut/train
Validation path: /kaggle/input/datasets/danushamsc25/sinfunddonut/SinFundDonut/validation


In [8]:
# ============================================================
# CELL 2 — DATA LOADING + DONUT SCHEMA CONVERSION
# ============================================================
# THIS CELL FIXES THE ROOT CAUSE DESCRIBED IN Guide/README.md /
# Guide/learnings.md.
#
# The raw SinFundDonut ground truth looks like:
#   {"gt_parse": {"<arbitrary Sinhala field-label text>": "<value>", ...}}
# i.e. the literal, highly variable form-field label is used AS a
# JSON key. If we fed this straight into Donut's json2token(), every
# unique field label would become its OWN special XML tag
# (<s_arbitrary Sinhala text>). With ~100 documents pulled from many
# different form types, most of those tags occur once — the model
# can never learn to *generate* a tag it saw zero or one times, so
# generation collapses into malformed output and token2json parses
# to nothing -> 0% metrics. This is a data-schema problem, not just
# a missing-token problem.
#
# The fix: keep the schema (tag names) FIXED and small
# ("form" / "question" / "answer"), and push all the variable
# Sinhala text into tag *content* instead of tag *names* — exactly
# how Donut's official CORD / FUNSD fine-tuning tasks are built:
#   {"gt_parse": {"form": [{"question": "<label text>",
#                            "answer": "<value text>"}, ...]}}

print("=" * 70)
print("DATASET")
print("=" * 70)
print("Dataset root:", DATASET_ROOT)
print("Train:", TRAIN_PATH)
print("Validation:", VALIDATION_PATH)


def load_split(split_path):
    jsonl_path = os.path.join(split_path, "metadata.jsonl")
    if not os.path.exists(jsonl_path):
        raise FileNotFoundError(f"metadata.jsonl not found at: {jsonl_path}")

    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            gt_raw = row["ground_truth"]
            gt_obj = json.loads(gt_raw) if isinstance(gt_raw, str) else gt_raw
            flat_kv = gt_obj.get("gt_parse", gt_obj)
            records.append(
                {
                    "image_path": os.path.join(split_path, row["file_name"]),
                    "flat_kv": flat_kv,
                }
            )
    return records


def flatten_to_form_schema(flat_kv):
    """Converts {"<label text>": "<value>" | [values...]} into the
    fixed Donut schema {"form": [{"question": ..., "answer": ...}]}.
    A list-valued answer (e.g. a multi-line address) expands into one
    form entry per value, reusing the same question text — this
    mirrors how FUNSD links one question box to multiple answer
    boxes."""
    entries = []
    for question_text, answer_value in flat_kv.items():
        question_text = str(question_text).strip()
        if isinstance(answer_value, list):
            for v in answer_value:
                entries.append({"question": question_text, "answer": str(v).strip()})
        else:
            entries.append({"question": question_text, "answer": str(answer_value).strip()})
    return {"form": entries}


def json2token(obj, sort_json_key=True):
    """Donut's official ground-truth -> XML-tag-sequence conversion
    (matches donut/model.py so that DonutProcessor.token2json, used
    at evaluation time, can invert it correctly)."""
    if isinstance(obj, dict):
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"]
        keys = sorted(obj.keys(), reverse=True) if sort_json_key else list(obj.keys())
        output = ""
        for k in keys:
            output += f"<s_{k}>" + json2token(obj[k], sort_json_key) + f"</s_{k}>"
        return output
    elif isinstance(obj, list):
        return "<sep/>".join(json2token(item, sort_json_key) for item in obj)
    else:
        return str(obj)


def collect_schema_tags(obj, tags):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k != "text_sequence":
                tags.add(k)
            collect_schema_tags(v, tags)
    elif isinstance(obj, list):
        for item in obj:
            collect_schema_tags(item, tags)


print("\nLoading train records...")
train_records = load_split(TRAIN_PATH)

print("Loading validation records...")
val_records = load_split(VALIDATION_PATH)

for rec in train_records + val_records:
    rec["schema"] = {"gt_parse": flatten_to_form_schema(rec["flat_kv"])}

schema_tags = set()
for rec in train_records + val_records:
    collect_schema_tags(rec["schema"], schema_tags)

print("\nDiscovered structural tags (should be small & fixed):", sorted(schema_tags))
if len(schema_tags) > 10:
    print(
        "WARNING: expected a handful of fixed tags (gt_parse/form/"
        "question/answer). A large number here means the raw label "
        "text is leaking into the schema again — check flatten_to_form_schema()."
    )

ADDITIONAL_SPECIAL_TOKENS = []
for t in sorted(schema_tags):
    ADDITIONAL_SPECIAL_TOKENS += [f"<s_{t}>", f"</s_{t}>"]
ADDITIONAL_SPECIAL_TOKENS.append("<sep/>")

print("\nSpecial tokens to add to the tokenizer:", ADDITIONAL_SPECIAL_TOKENS)

print("\nExample converted ground truth (train[0]):")
print(json.dumps(train_records[0]["schema"], ensure_ascii=False, indent=2)[:1500])

print(
    "\nTraining records:",
    len(train_records),
    "| Validation records:",
    len(val_records),
)
print("=" * 70)


DATASET
Dataset root: /kaggle/input/datasets/danushamsc25/sinfunddonut/SinFundDonut
Train: /kaggle/input/datasets/danushamsc25/sinfunddonut/SinFundDonut/train
Validation: /kaggle/input/datasets/danushamsc25/sinfunddonut/SinFundDonut/validation

Loading train records...
Loading validation records...

Discovered structural tags (should be small & fixed): ['answer', 'form', 'gt_parse', 'question']

Special tokens to add to the tokenizer: ['<s_answer>', '</s_answer>', '<s_form>', '</s_form>', '<s_gt_parse>', '</s_gt_parse>', '<s_question>', '</s_question>', '<sep/>']

Example converted ground truth (train[0]):
{
  "gt_parse": {
    "form": [
      {
        "question": "3 දිස්ත්‍රික්කය",
        "answer": "කලුතර"
      },
      {
        "question": "4. දුරකථන අංකය",
        "answer": "011-4568799"
      },
      {
        "question": "5. පොහොර වර්ගය",
        "answer": "සියල්ලම"
      },
      {
        "question": "6. පොහොරෙහි ස්වභාවය",
        "answer": "නැවුම"
      },
      {
        

In [ ]:
# ============================================================
# CELL 3 — TOKENIZER + MODEL BUILD (variant-aware)
# ============================================================

from transformers import (
    AutoTokenizer,
    DonutProcessor,
    VisionEncoderDecoderConfig,
    VisionEncoderDecoderModel,
)

print("=" * 70)
print(f"BUILDING MODEL (MODEL_VARIANT={MODEL_VARIANT!r})")
print("=" * 70)

print("\nLoading Donut processor (image processor + default tokenizer)...")
processor = DonutProcessor.from_pretrained(DONUT_BASE_ID, use_fast=False)

if MODEL_VARIANT == "donut_native":
    # No tokenizer swap: keep Donut-base's original multilingual
    # sentencepiece tokenizer. Byte-fallback can represent Sinhala
    # UTF-8 bytes, but the tokenizer was never trained on Sinhala, so
    # segmentation is inefficient. Use this only as a fast sanity
    # check of the training loop itself.
    model = VisionEncoderDecoderModel.from_pretrained(DONUT_BASE_ID)

elif MODEL_VARIANT == "donut_sinhala_vocab":
    # RECOMMENDED DEFAULT.
    # Keep Donut's own pretrained Swin encoder AND its own pretrained
    # decoder, so the encoder<->decoder cross-attention weights are
    # the ones Donut actually learned during large-scale pretraining.
    # Only the tokenizer/vocabulary is swapped for a Sinhala-aware one.
    print("Loading SinBERT tokenizer for Sinhala vocabulary...")
    text_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=False, token=HF_TOKEN)
    processor.tokenizer = text_tokenizer
    model = VisionEncoderDecoderModel.from_pretrained(DONUT_BASE_ID)

elif MODEL_VARIANT == "hybrid":
    # Option B ("Lego") from Guide/README.md: swap in a fully separate
    # Sinhala TrOCR decoder. Higher ceiling, higher risk — its
    # cross-attention was pretrained against a *different* encoder, so
    # it must learn to "look" at Donut's Swin features from scratch
    # using only a few dozen training images.
    print("Loading SinBERT tokenizer...")
    text_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=False, token=HF_TOKEN)
    processor.tokenizer = text_tokenizer

    donut_model = VisionEncoderDecoderModel.from_pretrained(DONUT_BASE_ID)
    donut_encoder = donut_model.encoder

    print("Loading custom TrOCR model:", TROCR_HUB_ID)
    trocr_model = VisionEncoderDecoderModel.from_pretrained(TROCR_HUB_ID, token=HF_TOKEN)
    trocr_decoder = trocr_model.decoder

    encoder_hidden_size = donut_encoder.config.hidden_size
    decoder_hidden_size = trocr_decoder.config.hidden_size
    print("Donut encoder hidden size:", encoder_hidden_size)
    print("TrOCR decoder hidden size:", decoder_hidden_size)

    hybrid_config = VisionEncoderDecoderConfig.from_encoder_decoder_configs(
        donut_encoder.config, trocr_decoder.config
    )
    model = VisionEncoderDecoderModel(config=hybrid_config, encoder=donut_encoder, decoder=trocr_decoder)

    if encoder_hidden_size != decoder_hidden_size:
        print(f"Creating encoder -> decoder projection: {encoder_hidden_size} -> {decoder_hidden_size}")
        model.enc_to_dec_proj = torch.nn.Linear(encoder_hidden_size, decoder_hidden_size)

    del donut_model, trocr_model
    gc.collect()
    torch.cuda.empty_cache()

else:
    raise ValueError(f"Unknown MODEL_VARIANT: {MODEL_VARIANT!r}")

# ------------------------------------------------------------
# DISABLE WORD-EMBEDDING TYING (all variants)
# ------------------------------------------------------------
# donut-base's own released checkpoint stores decoder.model.decoder.
# embed_tokens.weight and decoder.lm_head.weight as two SEPARATE
# tensors even though its config claims tie_word_embeddings=True
# (confirmed by a real Kaggle run: loading DONUT_BASE_ID prints
# "...both are present in the checkpoints, so we will NOT tie them.
# You should update the config with tie_word_embeddings=False").
# If we leave the config's claim as True, trainer.save_model()
# assumes lm_head.weight is a redundant duplicate of the tied
# embedding and skips writing it to disk — then reloading the
# checkpoint (load_best_model_at_end) reports
# "missing keys: ['decoder.lm_head.weight']" and silently falls back
# to a re-derived weight instead of the actually-trained one,
# corrupting the output projection of every reloaded/evaluated
# checkpoint. This was previously only set for the hybrid variant;
# a real run showed it is required for every variant that goes
# through DONUT_BASE_ID (see Guide/learnings.md, Session 3).
model.config.tie_word_embeddings = False

# ------------------------------------------------------------
# ENSURE PAD/EOS TOKENS EXIST
# ------------------------------------------------------------
# SinBERT (used in the donut_sinhala_vocab and hybrid variants) is a
# BERT-family tokenizer and, unlike Donut's original tokenizer, does
# not define an eos_token by default. Every part of this pipeline
# (target-sequence construction, generation stopping condition,
# output cleanup) assumes a real eos_token exists — without this
# guard, MODEL_VARIANT in {"donut_sinhala_vocab", "hybrid"} would
# crash later with a cryptic TypeError instead of failing here with
# a clear reason.

_special_tokens_to_add = {}
if processor.tokenizer.eos_token is None:
    _special_tokens_to_add["eos_token"] = "</s>"
if processor.tokenizer.pad_token is None:
    _special_tokens_to_add["pad_token"] = "<pad>"

if _special_tokens_to_add:
    print("\nTokenizer is missing standard special tokens; adding:", _special_tokens_to_add)
    processor.tokenizer.add_special_tokens(_special_tokens_to_add)

# ------------------------------------------------------------
# ADD DONUT STRUCTURAL TOKENS (discovered in Cell 2)
# ------------------------------------------------------------

num_added = processor.tokenizer.add_tokens(ADDITIONAL_SPECIAL_TOKENS)
print(f"\nAdded {num_added} structural tokens:", ADDITIONAL_SPECIAL_TOKENS)

model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.vocab_size = len(processor.tokenizer)

# ------------------------------------------------------------
# TASK PROMPT / DECODER START TOKEN
# ------------------------------------------------------------
# Following Donut's own convention: the task-start token
# ("<s_gt_parse>") is BOTH the decoder_start_token AND the first
# token of every target label sequence (see Cell 4). This lets
# generation condition on a task-specific state instead of a
# generic [CLS]/[BOS] embedding that has nothing to do with this task.

task_start_token_id = processor.tokenizer.convert_tokens_to_ids(TASK_START_TOKEN)
if task_start_token_id is None or task_start_token_id == processor.tokenizer.unk_token_id:
    raise ValueError(f"{TASK_START_TOKEN} was not added to the tokenizer vocabulary.")

model.config.decoder_start_token_id = task_start_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id

# ------------------------------------------------------------
# IMAGE SIZE
# ------------------------------------------------------------

model.config.encoder.image_size = IMAGE_SIZE
processor.image_processor.size = {"height": IMAGE_SIZE[0], "width": IMAGE_SIZE[1]}

# ------------------------------------------------------------
# GRADIENT CHECKPOINTING
# ------------------------------------------------------------

model.gradient_checkpointing_enable()
model.config.use_cache = False

model = model.cuda()

print("\n" + "=" * 70)
print("MODEL READY")
print("=" * 70)
print("Model variant:", MODEL_VARIANT)
print("Tokenizer vocabulary size:", len(processor.tokenizer))
print("Decoder start token:", TASK_START_TOKEN, "-> id:", task_start_token_id)
print("Image size:", IMAGE_SIZE)
print("Model device:", next(model.parameters()).device)
print("=" * 70)


In [10]:
# ============================================================
# CELL 4 — MAX_LENGTH SELECTION + PYTORCH DATASET +
#          PRE-TRAINING FORWARD/BACKWARD TEST
# ============================================================

from PIL import Image
from torch.utils.data import Dataset as TorchDataset

print("=" * 70)
print("DETERMINING MAX_LENGTH FROM ACTUAL DATA")
print("=" * 70)
# The previous version of this notebook hardcoded MAX_LENGTH (256 in
# one cell, 512 in another) without checking whether real target
# sequences fit. A truncated label silently corrupts training data
# (it can be cut off mid-tag). We measure the real distribution
# instead of guessing.
#
# A real Kaggle run (see Guide/learnings.md, Session 2) showed
# SinBERT target sequences run much longer than expected for this
# dataset: min=59, mean=516, max=1170 tokens. The original 768 hard
# cap here was silently truncating the longest document(s). Raised
# to 1536 — comfortably above the observed max, still cheap on a T4
# since most documents are far shorter and generation stops at EOS.

_target_strings = [
    json2token(rec["schema"]) + processor.tokenizer.eos_token for rec in train_records + val_records
]
_token_lengths = [len(processor.tokenizer(t, add_special_tokens=False)["input_ids"]) for t in _target_strings]

_max_observed = max(_token_lengths)
_HARD_CAP = 1536
MAX_LENGTH = int(min(_HARD_CAP, max(256, np.ceil((_max_observed + 8) / 32.0) * 32)))

print("Target length stats (tokens): min=%d, mean=%.1f, max=%d" % (
    min(_token_lengths), float(np.mean(_token_lengths)), _max_observed
))
print("Using MAX_LENGTH =", MAX_LENGTH)
if _max_observed + 8 > MAX_LENGTH:
    print(
        f"WARNING: MAX_LENGTH capped at {_HARD_CAP} — the longest "
        "document(s) will be truncated. Raise _HARD_CAP above if GPU "
        "memory allows and this warning still fires."
    )


class DonutDataset(TorchDataset):
    def __init__(self, records, processor, max_length):
        self.records = records
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image = Image.open(rec["image_path"]).convert("RGB")

        target_sequence = json2token(rec["schema"]) + self.processor.tokenizer.eos_token

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)

        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )["input_ids"].squeeze(0)

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}


train_dataset = DonutDataset(train_records, processor, MAX_LENGTH)
validation_dataset = DonutDataset(val_records, processor, MAX_LENGTH)

print("\nPyTorch datasets created:")
print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))

# ------------------------------------------------------------
# INSPECT ONE SAMPLE
# ------------------------------------------------------------

sample = train_dataset[0]
print("\nSample 0:")
print("pixel_values:", sample["pixel_values"].shape, sample["pixel_values"].dtype)
print("labels:", sample["labels"].shape, sample["labels"].dtype)

display_labels = sample["labels"].clone()
display_labels[display_labels == -100] = processor.tokenizer.pad_token_id
decoded = processor.tokenizer.decode(display_labels, skip_special_tokens=False)
print("\nDecoded target:")
print(decoded[:1500])

# ------------------------------------------------------------
# PRE-TRAINING FORWARD/BACKWARD TEST
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RUNNING PRE-TRAINING FORWARD/BACKWARD TEST")
print("=" * 70)

model.train()

pixel_values = sample["pixel_values"].unsqueeze(0).cuda()
labels = sample["labels"].unsqueeze(0).cuda()

outputs = model(pixel_values=pixel_values, labels=labels)
loss = outputs.loss
print("Forward pass successful. Loss:", loss.item())

loss.backward()
print("Backward pass successful.")

del pixel_values, labels, outputs, loss
torch.cuda.empty_cache()

print("=" * 70)
print("PRE-TRAINING TEST PASSED")
print("=" * 70)


DETERMINING MAX_LENGTH FROM ACTUAL DATA
Target length stats (tokens): min=59, mean=516.1, max=1170
Using MAX_LENGTH = 1184

PyTorch datasets created:
Training samples: 80
Validation samples: 20

Sample 0:
pixel_values: torch.Size([3, 960, 640]) torch.float32
labels: torch.Size([1184]) torch.int64

Decoded target:
<s_gt_parse><s_form><s_question>3 දිස්ත්‍රික්කය</s_question><s_answer>කලුතර</s_answer><sep/><s_question>4. දුරකථන අංකය</s_question><s_answer>011-4568799</s_answer><sep/><s_question>5. පොහොර වර්ගය</s_question><s_answer>සියල්ලම</s_answer><sep/><s_question>6. පොහොරෙහි ස්වභාවය</s_question><s_answer>නැවුම</s_answer><sep/><s_question>7. සාම්පල් කල දිනය</s_question><s_answer>2024-01-03</s_answer><sep/><s_question>8. සාම්පලයේ බර</s_question><s_answer>100g</s_answer><sep/><s_question>9. සාම්පල් අංකය</s_question><s_answer>01458</s_answer><sep/><s_question>11. විශ්ලෙෂණයට සුදුසු තත්වයේ පවතීද යන්න-</s_question><s_answer>ඔව්</s_answer><sep/><s_question>1. නම</s_question><s_answer>මුදියන්සේල

In [11]:
# ============================================================
# CELL 5 — METRICS (shared by Trainer eval loop and Cell 7)
# ============================================================
# Uses processor.token2json (the official, tested inverse of
# json2token) instead of ad hoc regex/json.loads, which is what the
# previous version of this notebook did and why parsing silently
# failed on any malformed generation.

import re
from collections import Counter

print("=" * 70)
print("METRICS SETUP")
print("=" * 70)


def clean_generated_text(text):
    text = text.replace(processor.tokenizer.eos_token, "")
    text = text.replace(processor.tokenizer.pad_token, "")
    # Strip everything up to and including the leading task-start
    # tag; token2json parses the remainder generically.
    text = re.sub(r"^.*?" + re.escape(TASK_START_TOKEN), "", text, count=1)
    text = text.replace(TASK_END_TOKEN, "")
    return text.strip()


def parse_form_entries(text):
    cleaned = clean_generated_text(text)
    try:
        parsed = processor.token2json(cleaned)
    except Exception:
        parsed = {}

    entries = parsed.get("form", []) if isinstance(parsed, dict) else []
    if isinstance(entries, dict):
        entries = [entries]

    pairs = []
    for e in entries:
        if isinstance(e, dict):
            q = str(e.get("question", "")).strip()
            a = str(e.get("answer", "")).strip()
            pairs.append((q, a))
    return pairs


def levenshtein(a, b):
    if a == b:
        return 0
    la, lb = len(a), len(b)
    if la == 0:
        return lb
    if lb == 0:
        return la
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * lb
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[lb]


def normalized_similarity(pred_pairs, ref_pairs):
    pred_str = "\n".join(f"{q}: {a}" for q, a in sorted(pred_pairs))
    ref_str = "\n".join(f"{q}: {a}" for q, a in sorted(ref_pairs))
    dist = levenshtein(pred_str, ref_str)
    denom = max(len(pred_str), len(ref_str), 1)
    return 1.0 - (dist / denom)


def score_predictions(pred_texts, ref_texts):
    """Field-level precision/recall/F1 over (question, answer) pairs,
    exact whole-document match rate, and a normalized edit-distance
    similarity (a smoother, partial-credit signal — Donut's paper
    reports an analogous "accuracy" via tree edit distance)."""
    matched = 0
    pred_total = 0
    ref_total = 0
    exact_matches = 0
    similarities = []

    for pred_text, ref_text in zip(pred_texts, ref_texts):
        pred_pairs = parse_form_entries(pred_text)
        ref_pairs = parse_form_entries(ref_text)

        pred_counter = Counter(pred_pairs)
        ref_counter = Counter(ref_pairs)
        intersection = pred_counter & ref_counter

        matched += sum(intersection.values())
        pred_total += len(pred_pairs)
        ref_total += len(ref_pairs)

        if pred_counter == ref_counter:
            exact_matches += 1

        similarities.append(normalized_similarity(pred_pairs, ref_pairs))

    precision = matched / pred_total if pred_total else 0.0
    recall = matched / ref_total if ref_total else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    exact_match_rate = exact_matches / len(pred_texts) if pred_texts else 0.0
    mean_similarity = float(np.mean(similarities)) if similarities else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "exact_match": exact_match_rate,
        "edit_similarity": mean_similarity,
    }


def compute_metrics(eval_preds):
    """Hooked into Seq2SeqTrainer (predict_with_generate=True) so
    every epoch's eval prints real field-level metrics instead of
    only eval_loss.

    Both predictions AND labels must have -100 replaced before
    decoding. Labels use -100 as the "ignore in loss" sentinel by
    construction (Cell 4). Predictions can *also* contain -100: when
    Trainer's evaluation_loop concatenates generated sequences from
    different eval batches that ended up different lengths (some
    documents hit EOS sooner than others), it pads the shorter ones
    using `nested_concat(..., padding_index=-100)` — a generic
    ignore-index convention, not something specific to labels. A
    real Kaggle run confirmed this: decoding pred_ids without this
    sanitization crashed with
    `OverflowError: out of range integral type conversion attempted`
    inside the fast tokenizer's Rust decode() the moment an eval
    batch's generated length differed from another's (see
    Guide/learnings.md, Session 2)."""
    pred_ids, label_ids = eval_preds
    if isinstance(pred_ids, tuple):
        pred_ids = pred_ids[0]

    pred_ids = np.where(pred_ids < 0, processor.tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

    pred_texts = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=False)
    ref_texts = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=False)

    return score_predictions(pred_texts, ref_texts)


print("Metrics ready: precision, recall, f1, exact_match, edit_similarity")
print("=" * 70)


METRICS SETUP
Metrics ready: precision, recall, f1, exact_match, edit_similarity


In [ ]:
# ============================================================
# CELL 6 — TRAINING
# ============================================================
# Fixes vs. the previous version of this notebook:
#  - generation_max_length was never set, so Seq2SeqTrainer fell
#    back to model.config.max_length (often 20!) during in-training
#    eval, silently truncating every generated sequence to garbage.
#    Now explicitly set to the data-driven MAX_LENGTH from Cell 4.
#  - compute_metrics is wired in, so eval_f1 / eval_edit_similarity
#    show up in the training logs every epoch (needed to see whether
#    the model is actually learning, not just that eval_loss drops).
#  - 80 training examples with batch_size=1, grad_accum=8, 10 epochs
#    was only ~100 optimizer steps total — too few for a seq2seq
#    model to learn a brand-new generation task. Raised epochs and
#    batch size for far more gradient updates on the same data.
#  - load_best_model_at_end + EarlyStoppingCallback so we keep the
#    checkpoint with the best validation F1, not just the last one.
#
# Session 3 fix (see Guide/learnings.md): a real run showed training
# loss and validation loss falling smoothly and continuously for 9
# straight epochs (61.5 -> 17.4 train, 13.4 -> 4.3 val) with NO sign
# of plateauing, while every task metric (F1, exact_match, and even
# edit_similarity) stayed at a hard 0.0 the entire time. This is
# expected early behavior, not a plateau: as long as generation
# produces zero parseable (question, answer) pairs, edit_similarity
# also floors at exactly 0 (Levenshtein distance from an empty
# prediction to a non-empty reference is just the reference's
# length, i.e. the worst possible score, not partial credit) — so
# these task metrics give literally no signal until the model first
# crosses the "produces syntactically valid tag output" threshold.
# metric_for_best_model="f1" + EarlyStoppingCallback(patience=8) was
# watching that frozen-at-zero signal and stopped training at epoch
# 9, before the model had any chance to cross that threshold, even
# though the loss curve gave no reason to stop.
#
# Fix: select the best checkpoint / drive early stopping off
# eval_loss instead (smooth and informative from epoch 1), with a
# much larger patience and epoch budget so the still-improving loss
# curve gets room to keep going. Task metrics (F1/edit_similarity/
# exact_match) are still computed and logged every epoch for
# visibility — they are just no longer what decides when to stop.

from transformers import EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments

print("=" * 70)
print("TRAINING CONFIGURATION")
print("=" * 70)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=120,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    per_device_eval_batch_size=2,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    generation_num_beams=1,
    fp16=True,
    gradient_checkpointing=True,
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=5,
    report_to="none",
    dataloader_num_workers=2,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=20)],
)

steps_per_epoch = -(-len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps))

print("\n" + "=" * 70)
print("STARTING SINHALA DOCUMENT UNDERSTANDING TRAINING")
print("=" * 70)
print("Model variant:", MODEL_VARIANT)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(validation_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Effective batch size:", training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
print("Approx. optimizer steps per epoch:", steps_per_epoch)
print("Approx. total optimizer steps:", steps_per_epoch * training_args.num_train_epochs)
print("Generation max length (eval):", training_args.generation_max_length)
print("Early stopping patience (epochs):", 20, "| selection metric: eval_loss (lower is better)")
print("GPU:", torch.cuda.get_device_name(0))
print("=" * 70)

# NOTE ON RUNTIME: a real Kaggle run of 9 epochs took ~12 minutes on a
# T4, so 120 epochs should take roughly 2.5-3 hours — comfortably
# within a single Kaggle session. If a session still runs out of
# GPU-hours before this finishes, reduce num_train_epochs above and
# re-run, or record how long one epoch actually took in
# KaggleRanNotebook/logs.txt so the schedule can be retuned with real
# data (see Guide/rules.md).

train_result = trainer.train()

# ------------------------------------------------------------
# SAVE TRAINED MODEL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SAVING FINAL MODEL")
print("=" * 70)

trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

metrics = train_result.metrics
print("\nTraining metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print("\nGPU memory:")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print("Final model saved to:", FINAL_MODEL_PATH)


In [13]:
# ============================================================
# CELL 7 — FINAL VALIDATION EVALUATION + REPORT
# ============================================================
# Runs generation over the full validation set with the best
# checkpoint (already loaded into `model` via load_best_model_at_end
# in Cell 6), reports field-level metrics, prints qualitative
# examples, and writes a JSON report into FINAL_MODEL_PATH.
#
# Bring `validation_report.json` back alongside KaggleRanNotebook/
# logs.txt after each Kaggle run — it has the parsed prediction vs.
# reference pairs needed to diagnose exactly which fields are wrong.

print("=" * 70)
print("EVALUATING DONUT MODEL ON VALIDATION SET")
print("=" * 70)

model.eval()

pred_texts = []
ref_texts = []

print("\nGenerating predictions for validation samples...")

for idx in range(len(validation_dataset)):
    sample = validation_dataset[idx]
    pixel_values = sample["pixel_values"].unsqueeze(0).cuda()

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_length=MAX_LENGTH,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            decoder_start_token_id=model.config.decoder_start_token_id,
        )

    pred_text = processor.tokenizer.decode(generated_ids[0], skip_special_tokens=False)

    ref_labels = sample["labels"].clone()
    ref_labels[ref_labels == -100] = processor.tokenizer.pad_token_id
    ref_text = processor.tokenizer.decode(ref_labels, skip_special_tokens=False)

    pred_texts.append(pred_text)
    ref_texts.append(ref_text)

print(f"Generated {len(pred_texts)} predictions.")

final_metrics = score_predictions(pred_texts, ref_texts)

print("\n" + "=" * 50)
print("DONUT MODEL VALIDATION RESULTS")
print("=" * 50)
for k, v in final_metrics.items():
    print(f"{k:16s}: {v:.4f}")
print("-" * 50)
print(f"Evaluated across {len(validation_dataset)} validation samples.")
print("=" * 50)

print("\nQualitative samples (first 3):")
for i in range(min(3, len(validation_dataset))):
    print(f"\n--- Sample {i} ---")
    print("Predicted pairs:", parse_form_entries(pred_texts[i]))
    print("Reference pairs:", parse_form_entries(ref_texts[i]))

report = {
    "model_variant": MODEL_VARIANT,
    "max_length": MAX_LENGTH,
    "num_validation_samples": len(validation_dataset),
    "metrics": final_metrics,
    "samples": [
        {
            "index": i,
            "predicted": parse_form_entries(pred_texts[i]),
            "reference": parse_form_entries(ref_texts[i]),
        }
        for i in range(len(validation_dataset))
    ],
}

os.makedirs(FINAL_MODEL_PATH, exist_ok=True)
report_path = os.path.join(FINAL_MODEL_PATH, "validation_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\nSaved detailed validation report to:", report_path)
print("=" * 70)


EVALUATING DONUT MODEL ON VALIDATION SET

Generating predictions for validation samples...
Generated 20 predictions.

DONUT MODEL VALIDATION RESULTS
precision       : 0.0000
recall          : 0.0000
f1              : 0.0000
exact_match     : 0.0000
edit_similarity : 0.0000
--------------------------------------------------
Evaluated across 20 validation samples.

Qualitative samples (first 3):

--- Sample 0 ---
Predicted pairs: []
Reference pairs: [('පදිංචි', 'කහටගස්දිගිලිය , සමගි මාවත , අංක 23 නිවසේ'), ('ශ්\u200dරේණියේ / වසරේ .', '7 වන'), ('ආබාධිත', 'පෝලියෝ'), ('.පාසලේ', 'කහටගස්දිගිලිය මහසෙන් කණිෂ්ට විද්\u200dයාලය'), ('නම', 'සුමුදු චන්ද්\u200dරසූරිය'), ('නම', 'අමල් සිල්වා'), ('දිනය', '2023 / 08 / 28'), ('දිනය', '2023 / 08 / 18'), ('අත්සන', '<signature>'), ('ආදායම රු', '7,000'), ('ශිෂ්\u200dයයාව / ශිෂ්\u200dයාව .', 'මල්ලව ආරච්චිගේ දොන් කවිදු ඉදුසර'), ('ශිෂ්\u200dයයා / ශිෂ්\u200dයාව', 'මල්ලව ආරච්චිගේ දොන් කවිදු ඉදුසර'), ('ශ්\u200dරේණියේ / වසරේ', '7 වන'), ('පමණ දුරක්', 'කි.මී 10 ක')]

--